In [16]:
#Importing libraries

import dash
from dash import dcc, html, Input, Output, dash_table, State
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
from dash.exceptions import PreventUpdate
from datetime import datetime
from dash.exceptions import PreventUpdate


In [2]:
# Load your preprocessed data
airqualitydata_preprocessed = pd.read_csv('cleaned_air_quality.csv')

In [3]:
airqualitydata_preprocessed.head()

,index,state_code,county_code,site_number,parameter_code,poc,parameter,si_id,method_code,method,...,change_quarter,days_since_change,change_month_sin,change_month_cos,method_complexity,equipment_type,summer_highs,prev_day_concentration,site_percentile,county_volatility
0,0,1,3,10,44201,1,Ozone,7,87,INSTRUMENTAL-ULTRA VIOLET ABSORPTION,...,3,275,-0.500000,-0.866025,3,Analytical,False,NaN,0.500000,0.438015
1,1,1,3,10,44201,1,Ozone,7,87,INSTRUMENTAL-ULTRA VIOLET ABSORPTION,...,2,9,0.866025,-0.500000,3,Analytical,False,2.09,0.500000,0.528864
2,2,1,49,9991,44201,1,Ozone,96297,47,INSTRUMENTAL-ULTRA VIOLET,...,4,166,-0.500000,0.866025,2,Analytical,False,NaN,0.520000,9.002969
3,3,1,51,4,44201,1,Ozone,104232,87,INSTRUMENTAL-ULTRA VIOLET ABSORPTION,...,3,275,-0.500000,-0.866025,3,Analytical,False,NaN,0.574627,11.072043
4,4,1,51,4,44201,1,Ozone,104232,87,INSTRUMENTAL-ULTRA VIOLET ABSORPTION,...,2,4,0.866025,-0.500000,3,Analytical,False,2.09,0.574627,0.000000


In [24]:

# Data preprocessing
numeric_cols = ['lvl1_monitor_concentration', 'county_volatility', 'site_percentile']
airqualitydata_preprocessed[numeric_cols] = airqualitydata_preprocessed[numeric_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
airqualitydata_preprocessed['date'] = pd.to_datetime(airqualitydata_preprocessed['date_of_last_change'])  # Ensure datetime format

# Initialize Dash app
app = dash.Dash(__name__)
server = app.server

app.layout = html.Div([
    # Title and description
    html.Div([
        html.H1("Interactive Air Quality Dashboard", style={'textAlign': 'center'}),
        html.P("Explore air quality metrics across different locations and time periods", 
              style={'textAlign': 'center', 'color': '#7FDBFF'})
    ], style={'marginBottom': '30px'}),
    
    # Filters Row
    html.Div([
        # State Filter
        html.Div([
            html.Label("Select States:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='state-filter',
                options=[{'label': f"State {s}", 'value': s} for s in sorted(airqualitydata_preprocessed['state_code'].unique())],
                multi=True,
                placeholder='All States',
                style={'width': '100%'}
            )
        ], style={'width': '22%', 'display': 'inline-block', 'padding': '10px'}),
        
        # Parameter Filter
        html.Div([
            html.Label("Select Parameter:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='parameter-filter',
                options=[{'label': p, 'value': p} for p in airqualitydata_preprocessed['parameter'].unique()],
                value='Ozone',
                style={'width': '100%'}
            )
        ], style={'width': '22%', 'display': 'inline-block', 'padding': '10px'}),
        
        # Date Range Filter
        html.Div([
            html.Label("Date Range:", style={'fontWeight': 'bold'}),
            dcc.DatePickerRange(
                id='date-range',
                min_date_allowed=airqualitydata_preprocessed['date'].min(),
                max_date_allowed=airqualitydata_preprocessed['date'].max(),
                start_date=airqualitydata_preprocessed['date'].min(),
                end_date=airqualitydata_preprocessed['date'].max()
            )
        ], style={'width': '22%', 'display': 'inline-block', 'padding': '10px'}),
        
        # Concentration Threshold
        html.Div([
            html.Label("Concentration Threshold:", style={'fontWeight': 'bold'}),
            dcc.Slider(
                id='concentration-slider',
                min=0,
                max=airqualitydata_preprocessed['lvl1_monitor_concentration'].max(),
                value=airqualitydata_preprocessed['lvl1_monitor_concentration'].median(),
                marks={i: str(i) for i in range(0, int(airqualitydata_preprocessed['lvl1_monitor_concentration'].max())+1, 10)}
            )
        ], style={'width': '30%', 'display': 'inline-block', 'padding': '10px'})
    ], style={'margin': '20px 0', 'backgroundColor': '#f8f9fa', 'borderRadius': '10px', 'padding': '15px'}),
    
    # Main Visualizations
    html.Div([
        # Map and Time Series
        html.Div([
            dcc.Graph(id='geo-map', style={'height': '500px'}),
            dcc.Graph(id='time-trend', style={'height': '500px'})
        ], style={'display': 'flex', 'flexDirection': 'row'}),
        
        # Statistical Plots
        html.Div([
            dcc.Graph(id='violin-plot'),
            dcc.Graph(id='heatmap-plot')
        ], style={'display': 'flex', 'flexDirection': 'row'}),
        
        # Additional Interactive Plots
        html.Div([
            dcc.Graph(id='animated-scatter'),
            dcc.Graph(id='parallel-coords')
        ], style={'display': 'flex', 'flexDirection': 'row'}),
        
        # Equipment and Site Analysis
        html.Div([
            dcc.Graph(id='equipment-chart'),
            dcc.Graph(id='site-analysis')
        ], style={'display': 'flex', 'flexDirection': 'row'})
    ]),
    
    # Data Table and Export
    html.Div([
        html.H3("Filtered Data", style={'marginTop': '30px'}),
        html.Div([
            html.Button("Export to CSV", id='export-button', n_clicks=0),
            dcc.Download(id="download-dataframe-csv")
        ], style={'margin': '10px 0'}),
        dash_table.DataTable(
            id='data-table',
            columns=[{"name": i, "id": i} for i in airqualitydata_preprocessed.columns],
            page_size=10,
            style_table={'overflowX': 'auto'},
            style_cell={'textAlign': 'left', 'padding': '10px'},
            style_header={'backgroundColor': '#2c3e50', 'color': 'white'},
            filter_action="native",
            sort_action="native"
        )
    ], style={'margin': '40px 0', 'padding': '15px', 'backgroundColor': '#f8f9fa', 'borderRadius': '10px'}),
    
    # Hidden div for storing filtered data
    html.Div(id='filtered-data-store', style={'display': 'none'})
])

# Callback to update all visualizations
@app.callback(
    [Output('geo-map', 'figure'),
     Output('time-trend', 'figure'),
     Output('violin-plot', 'figure'),
     Output('heatmap-plot', 'figure'),
     Output('animated-scatter', 'figure'),
     Output('parallel-coords', 'figure'),
     Output('equipment-chart', 'figure'),
     Output('site-analysis', 'figure'),
     Output('data-table', 'data'),
     Output('filtered-data-store', 'children')],
    [Input('state-filter', 'value'),
     Input('parameter-filter', 'value'),
     Input('date-range', 'start_date'),
     Input('date-range', 'end_date'),
     Input('concentration-slider', 'value')]
)
def update_dashboard(selected_states, selected_param, start_date, end_date, concentration_threshold):
    # Filter data based on selections
    filtered_df = airqualitydata_preprocessed.copy()
    
    if selected_param:
        filtered_df = filtered_df[filtered_df['parameter'] == selected_param]
    
    if selected_states:
        filtered_df = filtered_df[filtered_df['state_code'].isin(selected_states)]
    
    filtered_df = filtered_df[
        (filtered_df['date'] >= start_date) & 
        (filtered_df['date'] <= end_date) &
        (filtered_df['lvl1_monitor_concentration'] >= concentration_threshold)
    ]
    
    if filtered_df.empty:
        raise PreventUpdate
    
    # 1. Interactive Geospatial Map with click events
    geo_fig = px.scatter_mapbox(
        filtered_df,
        lat='latitude',
        lon='longitude',
        color='lvl1_monitor_concentration',
        size='county_volatility',
        hover_name='county_name',
        hover_data=['site_number', 'parameter', 'date'],
        color_continuous_scale=px.colors.sequential.Plasma,
        zoom=5,
        title='Interactive Air Quality Map (click on points for details)',
        height=500
    )
    geo_fig.update_layout(
        mapbox_style="open-street-map",
        clickmode='event+select'
    )
    
    # 2. Time-Series with Range Selector
    trend_fig = px.line(
        filtered_df.groupby(['date', 'change_quarter'])['lvl1_monitor_concentration'].mean().reset_index(),
        x='date',
        y='lvl1_monitor_concentration',
        color='change_quarter',
        title='Concentration Trends with Range Selector',
        height=500
    )
    trend_fig.update_xaxes(
        rangeslider_visible=True,
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=6, label="6m", step="month", stepmode="backward"),
                dict(count=1, label="YTD", step="year", stepmode="todate"),
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        )
    )
    
    # 3. Interactive Violin Plot with Box Plot
    violin_fig = px.violin(
        filtered_df,
        y='lvl1_monitor_concentration',
        x='equipment_type',
        box=True,
        points="all",
        hover_data=['site_number', 'county_name'],
        title='Distribution by Equipment Type (hover for details)'
    )
    
    # 4. Heatmap of Concentration by Time and Location
    heatmap_fig = px.density_heatmap(
        filtered_df,
        x='date',
        y='county_name',
        z='lvl1_monitor_concentration',
        title='Concentration Heatmap by Date and County',
        height=400
    )
    
    # 5. Animated Scatter Plot
    animated_fig = px.scatter(
        filtered_df,
        x='change_year',
        y='lvl1_monitor_concentration',
        size='county_volatility',
        color='equipment_type',
        animation_frame='change_quarter',
        hover_name='county_name',
        title='Seasonal Changes in Air Quality (Animated)',
        height=400
    )
    
    # 6. Parallel Coordinates Plot
    parallel_fig = px.parallel_coordinates(
        filtered_df,
        dimensions=['lvl1_monitor_concentration', 'county_volatility', 'site_percentile', 'method_complexity'],
        color='lvl1_monitor_concentration',
        title='Multidimensional Analysis',
        height=400
    )
    
    # 7. Interactive Sunburst Chart
    equip_fig = px.sunburst(
        filtered_df,
        path=['state_code', 'county_name', 'equipment_type'],
        values='site_number',
        color='lvl1_monitor_concentration',
        title='Equipment Distribution Hierarchy',
        height=400
    )
    
    # 8. Site Analysis Scatter Plot
    site_fig = px.scatter(
        filtered_df,
        x='days_since_change',
        y='lvl1_monitor_concentration',
        color='county_name',
        size='county_volatility',
        hover_data=['site_number', 'method'],
        title='Site Performance Analysis',
        height=400
    )
    
    # Data Table
    table_data = filtered_df.to_dict('records')
    
    # Store filtered data for export
    stored_data = filtered_df.to_json(date_format='iso', orient='split')
    
    return geo_fig, trend_fig, violin_fig, heatmap_fig, animated_fig, parallel_fig, equip_fig, site_fig, table_data, stored_data

# Callback for exporting data
@app.callback(
    Output("download-dataframe-csv", "data"),
    Input("export-button", "n_clicks"),
    State('filtered-data-store', 'children'),
    prevent_initial_call=True
)
def export_data(n_clicks, stored_data):
    if n_clicks > 0 and stored_data:
        filtered_df = pd.read_json(stored_data, orient='split')
        return dcc.send_data_frame(filtered_df.to_csv, "filtered_air_quality_data.csv")
    raise PreventUpdate

# Run the app
if __name__ == '__main__':
    #app.run(jupyter_mode='inline', debug=True)  
    app.run(jupyter_mode='external', port=8050)# debug=True helps see errors

Dash app running on http://127.0.0.1:8050/
